In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

data = {
    "order_id": [1,2,3,4,5,5,7,8,9,10],
    "customer_name": ["John", "Alice", None, "Bob", "Eve", "Eve", "Charlie", "David", "Emma", "Frank"],
    "email": ["john@gmail.com", "alice@gmail.com", "noemail", "bob@gmail", None, None, "charlie@gmail.com", "david@gmail.com", "emma@gmail.com", "frank@gmail.com"],
    "order_date": ["2023-01-01", "01-02-2023", "2023/03/01", "2023-04-01", "wrong_date", "wrong_date", "2023-06-01", "2023-07-01", "2023-08-01", "2023-09-01"],
    "price": [100, 200, -50, 300, 400, 400, 500, None, 700, 800],
    "category": ["Electronics", "electronics", "Clothing", "clothing", "Electronics", "Electronics", "Clothing", "Clothing", "electronics", "Electronics"]
}

df = pd.DataFrame(data)
df.to_csv("raw_data.csv", index=False)

df.head()

,order_id,customer_name,email,order_date,price,category
0,1,John,john@gmail.com,2023-01-01,100.0,Electronics
1,2,Alice,alice@gmail.com,01-02-2023,200.0,electronics
2,3,None,noemail,2023/03/01,-50.0,Clothing
3,4,Bob,bob@gmail,2023-04-01,300.0,clothing
4,5,Eve,None,wrong_date,400.0,Electronics


### Data Quality Rules

### 1. order_id should be unique
### 2. email should contain '@' and '.'
### 3. price should be > 0
### 4. order_date should be valid date format
### 5. category should be standardized
### 6. No missing critical fields (customer_name, email)

# 🟢🟢 CLEANING PIPELINES

#### 1️⃣ Remove Duplicates

In [2]:
df = df.drop_duplicates()

### 2️⃣ Handle Missing Values

In [3]:
df["customer_name"] = df["customer_name"].fillna("Unknown")
df["price"] = df["price"].fillna(df["price"].median())

### 3️⃣ Fix Email Issues

In [4]:
df = df[df["email"].str.contains("@", na=False)]

### 4️⃣ Fix Date Format

In [5]:
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df = df.dropna(subset=["order_date"])

### 5️⃣ Fix Negative Prices

In [6]:
df = df[df["price"] > 0]

### 6️⃣ Standardize Categories

In [7]:
df["category"] = df["category"].str.lower()

### 7️⃣ Save Clean Data

In [10]:
df.to_csv("cleaned_data.csv", index=False)

In [11]:
from google.colab import files
files.download("cleaned_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# LOADING DATA INTO SQL

In [12]:
import sqlite3
import pandas as pd

# Load cleaned data
df = pd.read_csv("cleaned_data.csv")

# Create database
conn = sqlite3.connect("data_quality.db")

# Load into SQL table
df.to_sql("orders", conn, if_exists="replace", index=False)

# Verify
pd.read_sql("SELECT * FROM orders LIMIT 5", conn)

,order_id,customer_name,email,order_date,price,category
0,1,John,john@gmail.com,2023-01-01,100.0,electronics
1,4,Bob,bob@gmail,2023-04-01,300.0,clothing
2,7,Charlie,charlie@gmail.com,2023-06-01,500.0,clothing
3,8,David,david@gmail.com,2023-07-01,350.0,clothing
4,9,Emma,emma@gmail.com,2023-08-01,700.0,electronics


### Data Validation Queries

#### 1️⃣ Check for Duplicate Order IDs:

In [18]:
query = """
SELECT order_id, COUNT(*) as count
FROM orders
GROUP BY order_id
HAVING COUNT(*) > 1;
"""

pd.read_sql(query, conn)

,order_id,count


#### 2️⃣ Check for Missing Values:

In [21]:
query= """
select * from orders
where customer_name is null
or email is null
or price is null;
"""
pd.read_sql(query, conn)

,order_id,customer_name,email,order_date,price,category


#### 3️⃣ Check Invalid Emails:

In [22]:
query= """
select * from orders where email not like "%@%.%";
"""
pd.read_sql(query,conn)


,order_id,customer_name,email,order_date,price,category
0,4,Bob,bob@gmail,2023-04-01,300.0,clothing


#### 4️⃣ Check Negative or Zero Prices:

In [23]:
query= """
select * from orders
where price <=0;
"""
pd.read_sql(query,conn)

,order_id,customer_name,email,order_date,price,category


#### 5️⃣ Check Date Issues:

In [25]:
query= """
select * from orders
where order_date is null;
"""
pd.read_sql(query,conn)

,order_id,customer_name,email,order_date,price,category


#### 6️⃣ Check Category Standardization:

In [26]:
query="""
select distinct category from orders;
"""
pd.read_sql(query,conn)


,category
0,electronics
1,clothing
